# Part 04 — Unseen Eval, Fusion & Deployment

Satellite notebook split from `capstone_with_results.ipynb` to keep the master lighter to open.
Outputs/source sync automatically via `scripts/sync_notebook_parts.py` (run the final sync cell).


In [ ]:
# ── PART BOOTSTRAP: Part 04 — Unseen Eval, Fusion & Deployment ──
# Lightweight satellite notebook — syncs back into notebooks/capstone_with_results.ipynb
import sys
from pathlib import Path

def _find_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "src" / "vision_brain.py").is_file() and (base / "explanations" / "build_all.py").is_file():
            return base
        if (base / "src" / "vision_brain.py").is_file():
            return base
    return Path.cwd().parent if Path.cwd().name in {"notebooks", "parts"} else Path.cwd()

REPO_ROOT = _find_root()
SRC = REPO_ROOT / "src"
DATA = REPO_ROOT / "Data"
if not DATA.is_dir():
    DATA = REPO_ROOT / "data"
sys.path.insert(0, str(SRC))
sys.path.insert(0, str(REPO_ROOT / "scripts")) if (REPO_ROOT / "scripts").is_dir() else None

try:
    import torch
    TORCH_OK = True
except Exception:
    TORCH_OK = False

# Optional shared helpers used by several sections
try:
    from cm_helpers import plot_confusion_matrices_pair  # noqa: F401
except Exception:
    plot_confusion_matrices_pair = None

import numpy as np  # noqa: F401
import pandas as pd  # noqa: F401
import matplotlib.pyplot as plt  # noqa: F401

print(f"Part notebook ready | root={REPO_ROOT} | TORCH_OK={TORCH_OK} | DATA={DATA}")
print("When finished: run the last cell to sync outputs → capstone_with_results.ipynb")

### 10.1 · NLP Brain — TF-IDF Alert Scoring

Ontario 511 alerts are unstructured text. We tokenize, apply **TF-IDF**, and sum
weights on a hazard lexicon (ice, blizzard, collision, closed…) to produce **T ∈ [0, 1]**.


In [ ]:
# ── 10.1  NLP Brain (TF-IDF) ─────────────────────────────────────────────────
from nlp_brain import fit_tfidf, score_all_scenarios, SCENARIO_ALERTS, t_score_from_text

tfidf_vec = fit_tfidf()
nlp_rows = score_all_scenarios(tfidf_vec)

print("NLP Brain — TF-IDF hazard scores (T)")
print("=" * 70)
for name, snippet, t in nlp_rows:
    print(f"  {name[:42]:<42}  T={t:.3f}")
    print(f"    Alert: {snippet}")
print()

# Bar chart
import pandas as pd
df_nlp = pd.DataFrame(nlp_rows, columns=["Scenario", "Alert", "T_score"])
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(df_nlp["Scenario"], df_nlp["T_score"], color="#A23B72", edgecolor="white")
ax.set_xlim(0, 1.05)
ax.set_xlabel("T score (NLP hazard index)")
ax.set_title("NLP Brain — TF-IDF Risk per Ontario Highway Scenario", fontweight="bold")
ax.invert_yaxis()
for bar, val in zip(bars, df_nlp["T_score"]):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2, f"{val:.2f}", va="center")
plt.tight_layout()
plt.show()


### 10.2 · Safety Score Fusion (T + V + E → S)

$$S = (w_T \cdot T + w_V \cdot V + w_E \cdot E_{index}) \times 100$$

| Tier | S range | Recommended speed |
|------|---------|-------------------|
| LOW | 0–30 | 100% of posted limit |
| MEDIUM | 31–70 | 80% |
| HIGH | 71–100 | 60% |


In [ ]:
# ── 10.2  Safety Score fusion + dashboard ────────────────────────────────────
from safety_score import fuse_scenario, risk_tier, W_T, W_V, W_E

# Vision V-scores per scenario (from Section 6 if available, else scenario priors)
V_PRIORS = {
    "TC-1 Clear rush-hour (401 Jul 5pm)": 0.15,
    "TC-2 Blizzard night (Hwy400 Jan 2am)": 0.92,
    "TC-3 Wet dawn bicycle (Hwy7 Apr 6am)": 0.45,
    "TC-4 Clear Sunday (Hwy115 Jun 9am)": 0.10,
    "TC-5 Ice storm rush (QEW Feb 5pm)": 0.88,
}
if "vision_val_acc" in dir() and vision_val_acc is not None:
    print(f"Vision Brain trained (val acc {vision_val_acc:.1%}) — using scenario V priors for fusion demo.")

# Build fusion table from live test cases (TC dict from Section 8.6)
if "TC" not in globals():
    from nlp_brain import SCENARIO_ALERTS
    TC = {k: [17,7,3,0,1,0,0,1] for k in SCENARIO_ALERTS}  # fallback
    print("Note: using fallback TC — run Section 8.6 for full live test cases.")
if "nlp_rows" not in globals():
    from nlp_brain import fit_tfidf, score_all_scenarios
    nlp_rows = score_all_scenarios(fit_tfidf())

fusion_rows = []
for (scenario, feat), (_, _, t_sc) in zip(TC.items(), nlp_rows):
    occ_hour, month_num, season_num, is_night, is_rush, ped, bike, auto = feat
    v = V_PRIORS.get(scenario, 0.3)
    winter = season_num == 1 and month_num in (1, 2, 12)
    row = fuse_scenario(t_sc, v, month_num, season_num, is_night, is_winter_storm=winter)
    row["Scenario"] = scenario
    fusion_rows.append(row)

df_fusion = pd.DataFrame(fusion_rows)
cols = ["Scenario", "T_nlp", "V_vision", "E_index", "S", "tier", "V_rec_kmh"]
print("\n=== Safety Score Fusion (Sprint 3) ===")
print(df_fusion[cols].to_string(index=False))

# ── Dashboard prototype ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={"height_ratios": [1.2, 1]})

# Top: stacked brain contributions
x = np.arange(len(df_fusion))
w = 0.6
t_contrib = df_fusion["T_nlp"] * W_T * 100
v_contrib = df_fusion["V_vision"] * W_V * 100
e_contrib = df_fusion["E_index"] * W_E * 100
axes[0].bar(x, t_contrib, w, label="NLP (T)", color="#A23B72")
axes[0].bar(x, v_contrib, w, bottom=t_contrib, label="Vision (V)", color="#2E86AB")
axes[0].bar(x, e_contrib, w, bottom=t_contrib + v_contrib, label="Environment (E)", color="#F18F01")
axes[0].axhline(30, color="green", linestyle="--", alpha=0.7, label="Low/Med boundary")
axes[0].axhline(70, color="red", linestyle="--", alpha=0.7, label="Med/High boundary")
axes[0].set_xticks(x)
axes[0].set_xticklabels([s.split("(")[0].strip() for s in df_fusion["Scenario"]], rotation=15, ha="right")
axes[0].set_ylabel("Safety Score S")
axes[0].set_title("Smart-Shield Dashboard — Brain Contributions to Safety Score S", fontweight="bold")
axes[0].legend(loc="upper right", fontsize=8)
axes[0].set_ylim(0, 105)

# Bottom: recommended speed
colors = [risk_tier(s)[1] for s in df_fusion["S"]]
axes[1].barh(df_fusion["Scenario"], df_fusion["V_rec_kmh"], color=colors, edgecolor="white")
axes[1].axvline(100, color="gray", linestyle=":", label="Posted 100 km/h")
axes[1].set_xlabel("Recommended speed (km/h)")
axes[1].set_title("Dynamic Speed Recommendation by Scenario", fontweight="bold")
axes[1].legend()
plt.tight_layout()
plt.show()


### 10.3 · SHAP Explainability (Sprint 3 Ethics Deliverable)

**SHAP** (SHapley Additive exPlanations) shows how each feature pushes the
Random Forest prediction toward Fatal / Injury / PD-Only. Required for the
explainability row in the Ethics Risk Register.


In [ ]:
# ── 10.3  SHAP explainability for final RF model ─────────────────────────────
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

if "best_estimators" not in globals() or not best_estimators:
    raise RuntimeError("Run Section 8.3 GridSearchCV first (cell ~51).")

rf_model = best_estimators["Random Forest (Tuned)"]
feat_names = available if "available" in globals() else [f"f{i}" for i in range(X_test_sc.shape[1])]

SHAP_SAMPLE = min(500, len(X_test_sc))
rng = np.random.default_rng(42)
idx = rng.choice(len(X_test_sc), SHAP_SAMPLE, replace=False)
X_shap = X_test_sc[idx]
n_feat = X_shap.shape[1]


def _shap_for_class(shap_values, cls_idx: int, n_features: int) -> np.ndarray:
    """Align SHAP matrix to X columns (handles list/3D output + offset column)."""
    if isinstance(shap_values, list):
        sv = np.asarray(shap_values[cls_idx])
    elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        sv = shap_values[:, :, cls_idx]
    else:
        sv = np.asarray(shap_values)

    if sv.ndim == 2 and sv.shape[1] == n_features + 1:
        sv = sv[:, :-1]  # drop constant offset column (newer SHAP)
    if sv.ndim == 2 and sv.shape[1] > n_features:
        sv = sv[:, :n_features]
    return sv


print(f"Computing SHAP values for {SHAP_SAMPLE} test samples...")
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_shap)

class_names = ["PD-Only", "Injury", "Fatal"]
for cls_idx, cls_name in enumerate(class_names):
    sv = _shap_for_class(shap_values, cls_idx, n_feat)
    print(f"\n--- SHAP summary: class {cls_name} (shape {sv.shape}) ---")
    shap.summary_plot(
        sv, X_shap, feature_names=feat_names,
        show=False, max_display=len(feat_names),
    )
    plt.title(f"SHAP — Random Forest (Tuned) · Predicting {cls_name}", fontweight="bold")
    plt.tight_layout()
    plt.show()

fatal_sv = _shap_for_class(shap_values, 2, n_feat)
fatal_shap = np.abs(fatal_sv).mean(axis=0)
fi_shap = pd.Series(fatal_shap, index=feat_names[: len(fatal_shap)]).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 4))
fi_shap.plot(kind="barh", ax=ax, color="#C73E1D")
ax.set_title("Mean |SHAP| for Fatal class — Feature Impact", fontweight="bold")
ax.set_xlabel("Mean absolute SHAP value")
plt.tight_layout()
plt.show()


### 10.4 · Model Deployment

Serialize the tuned Random Forest + scaler for the Smart-Shield API / dashboard backend.


In [ ]:
# ── 10.4  Serialize models for deployment ────────────────────────────────────
import joblib
from nlp_brain import fit_tfidf
from pathlib import Path

MODEL_DIR = (REPO_ROOT / "models") if "REPO_ROOT" in globals() else Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

artifacts = {
    "rf_tuned": best_estimators["Random Forest (Tuned)"],
    "lr_tuned": best_estimators["Logistic Regression (Tuned)"],
    "scaler": scaler,
    "feature_names": available,
    "tfidf_vectorizer": tfidf_vec if "tfidf_vec" in globals() else fit_tfidf(),
}

paths = {}
for name, obj in artifacts.items():
    p = MODEL_DIR / f"{name}.joblib"
    joblib.dump(obj, p)
    paths[name] = str(p.resolve())
    print(f"  Saved {name} → {p}")

# Quick load test
rf_loaded = joblib.load(MODEL_DIR / "rf_tuned.joblib")
test_pred = rf_loaded.predict(X_test_sc[:5])
print(f"\nLoad test OK — sample preds: {test_pred}")

if TORCH_OK and "model_dnn" in dir() and model_dnn is not None:
    torch_path = MODEL_DIR / "dnn_smart_shield.pt"
    torch.save(model_dnn.state_dict(), torch_path)
    print(f"  Saved DNN weights → {torch_path}")

if TORCH_OK and "vision_model" in dir() and vision_model is not None:
    vis_path = MODEL_DIR / "vision_resnet18.pt"
    torch.save(vision_model.state_dict(), vis_path)
    print(f"  Saved Vision model → {vis_path}")

print("\n=== Sprint 3 Complete ===")
print("  ✓ NLP Brain (TF-IDF T scores)")
print("  ✓ Vision Brain (Section 6 V scores)")
print("  ✓ Safety Score fusion + dashboard")
print("  ✓ SHAP explainability")
print("  ✓ Models saved to ./models/")


In [ ]:
# ── PART SYNC → main notebook ──
import subprocess
import sys
from pathlib import Path

def _find_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "scripts" / "sync_notebook_parts.py").is_file():
            return base
    return Path.cwd()

_ROOT = _find_root()
rc = subprocess.run(
    [sys.executable, str(_ROOT / "scripts" / "sync_notebook_parts.py"),
     "--direction", "parts-to-main", "--only", "04_fusion_deploy.ipynb"],
    cwd=_ROOT,
).returncode
print("Synced to capstone_with_results.ipynb" if rc == 0 else f"Sync failed (exit {rc})")